# 3 — Anomaly Detection, Drift and Alerting

**Sensor Intelligence Platform** — analytical walkthrough (3 / 7)

Detecting that something changed is only half the job; an operator needs a **ranked, explained alert** and a signal for when the *model itself* has gone stale. This notebook covers the full detection-to-alert path.

1. Point anomalies (robust z-score) on a spiked, seasonal signal, scored against ground truth.
2. Regime shifts with the CUSUM change-point detector.
3. Distributional drift with the Population Stability Index.
4. Turning raw detections into severity-ranked, de-duplicated alerts.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

NAVY, ORANGE, TEAL, RED, GREY = "#1f3a5f", "#e8893b", "#2a9d8f", "#c0392b", "#9aa0ad"

## 3.1 Point-anomaly detection

We simulate temperature — a channel with a strong daily cycle — and inject two spikes. A rolling detector on the **raw** signal is blinded: the daily swing inflates the median/MAD spread so the spikes look ordinary. We therefore **deseasonalise** first — subtract the time-of-day profile estimated from the clean early days — and run the rolling median/MAD **robust z-score** detector on the residual. MAD-based scoring resists the very outliers it is meant to flag.

In [2]:
from sensor_intelligence.simulation import (
    SensorSimulator, SensorSpec, SimulationConfig, AnomalyInjection,
)
from sensor_intelligence.domain import TimeSeriesWindow
from sensor_intelligence.anomaly import RobustZScoreDetector, CusumChangePointDetector

PERIOD = 96
cfg = SimulationConfig(
    sensors=[SensorSpec('temperature', baseline=60.0, daily_amplitude=8.0,
                        noise_std=0.6, noise_ar=0.4, unit='C')],
    n_steps=PERIOD * 6, step_seconds=900, seed=33,
    anomalies=[
        AnomalyInjection('temperature', start_step=PERIOD * 2 + 40, magnitude=14.0),
        AnomalyInjection('temperature', start_step=PERIOD * 4 + 10, magnitude=-11.0),
    ],
)
frame = SensorSimulator(cfg).run()
window = TimeSeriesWindow(sensor_id='temperature',
                          timestamps=list(frame.timestamp),
                          values=[float(v) for v in frame.value])

# Deseasonalise: subtract the time-of-day profile learned from the clean early days.
minute = frame.timestamp.dt.hour * 60 + frame.timestamp.dt.minute
ref = frame.iloc[:PERIOD * 2]
profile = ref.groupby(ref.timestamp.dt.hour * 60 + ref.timestamp.dt.minute).value.mean()
residual = frame.value.to_numpy() - minute.map(profile).to_numpy()
resid_window = TimeSeriesWindow(sensor_id='temperature',
                                timestamps=list(frame.timestamp),
                                values=[float(v) for v in residual])

detector = RobustZScoreDetector(window_size=PERIOD, threshold=5.0)
raw_hits = detector.detect(window)
anomalies = detector.detect(resid_window)
print(f'on the raw signal:              {len(raw_hits)} flagged')
print(f'on the deseasonalised residual: {len(anomalies)} flagged')
for a in anomalies:
    print(f'  {a.timestamp:%Y-%m-%d %H:%M}  score={a.score:5.1f}  {a.reason_codes[0]}')

on the raw signal:              0 flagged
on the deseasonalised residual: 2 flagged
  2024-01-03 10:00  score= 29.7  modified z-score +29.66 exceeds 5
  2024-01-05 02:30  score= 16.4  modified z-score -16.44 exceeds 5


In [3]:
ts = pd.Series(window.values, index=pd.to_datetime(window.timestamps))
flagged = pd.to_datetime([a.timestamp for a in anomalies])
truth = frame.set_index('timestamp').is_anomaly
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts.index, ts.values, color=NAVY, lw=0.9, label='temperature')
ax.scatter(truth[truth].index, ts[truth[truth].index], color=RED, s=70,
           marker='o', facecolors='none', linewidths=1.6, label='ground-truth fault')
ax.scatter(flagged, ts[flagged], color=ORANGE, s=26, zorder=5, label='detected (residual)')
ax.set(title='Robust z-score on the deseasonalised residual vs. ground truth',
       xlabel='time', ylabel='C')
ax.legend(loc='upper left', fontsize=9)
fig.tight_layout()

### Detection quality
We score detections against the labels with a small tolerance (a detection counts as a hit if it lands within two samples of a labelled fault).

In [4]:
truth_idx = np.where(frame.is_anomaly.to_numpy())[0]
times = pd.Index(window.timestamps)
det_idx = [times.get_loc(a.timestamp) for a in anomalies]
tol = 2
tp = sum(any(abs(d - t) <= tol for t in truth_idx) for d in det_idx)
fp = len(det_idx) - tp
recovered = sum(any(abs(d - t) <= tol for d in det_idx) for t in truth_idx)
precision = tp / max(len(det_idx), 1)
recall = recovered / max(len(truth_idx), 1)
print(f'precision={precision:.2f}  recall={recall:.2f}  (tp={tp}, fp={fp})')

precision=1.00  recall=1.00  (tp=2, fp=0)


## 3.2 Change-point detection

Point detectors miss **sustained** shifts. The CUSUM detector accumulates standardized deviations and fires once per regime change, adapting its baseline afterward. We build an explicit level shift on a stationary series to demonstrate it cleanly.

In [5]:
rng = np.random.default_rng(7)
shift = np.concatenate([rng.normal(20, 0.5, 250), rng.normal(26, 0.5, 250)])
stamps = list(pd.date_range('2024-03-01', periods=shift.size, freq='15min'))
shift_window = TimeSeriesWindow(sensor_id='reactor', timestamps=stamps,
                                values=shift.tolist())

cps = CusumChangePointDetector(slack=0.5, threshold=8.0, reference_size=200).detect(shift_window)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(stamps, shift, color=NAVY, lw=0.9)
for cp in cps:
    ax.axvline(cp.timestamp, color=RED, ls='--', lw=1.5)
    ax.text(cp.timestamp, shift.max(), f'  {cp.reason_codes[0].split(" (")[0]}',
            color=RED, fontsize=9, va='top')
ax.set(title=f'CUSUM change-point detection ({len(cps)} change point)',
       xlabel='time', ylabel='value')
fig.tight_layout()

## 3.3 Distributional drift (PSI)

The **Population Stability Index** quantifies how far a current sample's distribution has moved from a reference. It is the standard early-warning signal that a deployed model's inputs — or its errors — no longer look like training time. Convention: PSI < 0.1 stable, 0.1–0.25 minor, > 0.25 major.

In [6]:
from sensor_intelligence.drift import population_stability_index, PsiDriftDetector

reference = rng.normal(0, 1, 4000)
shifts = np.linspace(0, 2.5, 12)
psis = [population_stability_index(reference, rng.normal(mu, 1, 4000)) for mu in shifts]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.hist(reference, bins=50, alpha=0.6, color=NAVY, label='reference')
a1.hist(rng.normal(2.0, 1, 4000), bins=50, alpha=0.6, color=ORANGE, label='shifted (+2.0)')
a1.set(title='Reference vs. drifted distribution', xlabel='value'); a1.legend(fontsize=9)
a2.plot(shifts, psis, 'o-', color=TEAL)
a2.axhline(0.1, color=GREY, ls='--', lw=1); a2.axhline(0.25, color=RED, ls='--', lw=1)
a2.text(0.02, 0.27, 'major', color=RED, fontsize=9)
a2.set(title='PSI grows with the mean shift', xlabel='mean shift', ylabel='PSI')
fig.tight_layout()
PsiDriftDetector(threshold=0.2).evaluate(reference, rng.normal(2.0, 1, 4000))

DriftResult(metric='psi', score=3.5194572334398995, threshold=0.2, drifted=True, detail='PSI 3.519 vs threshold 0.2')

## 3.4 From detections to alerts

`AlertPolicy` maps scores to severity bands and **merges** detections that share a sensor and timestamp into one alert (so a coincident spike and change point do not double-page an operator), carrying every reason code forward. Here the two spikes yield two critical alerts, which we render as an operator-facing Markdown report.

In [7]:
from sensor_intelligence.alerting import AlertPolicy
from sensor_intelligence.reporting import build_report

alerts = AlertPolicy().from_anomalies(anomalies)
table = pd.DataFrame([
    {'severity': a.severity.value, 'sensor': a.sensor_id,
     'time': a.timestamp.strftime('%Y-%m-%d %H:%M'),
     'methods': ', '.join(sorted({an.method for an in a.anomalies})),
     'message': a.message.split(': ', 1)[1]}
    for a in alerts
])
table

,severity,sensor,time,methods,message
0,critical,temperature,2024-01-03 10:00,robust_zscore,robust_zscore flagged sensor temperature (scor...
1,critical,temperature,2024-01-05 02:30,robust_zscore,robust_zscore flagged sensor temperature (scor...


In [8]:
print(build_report('Temperature monitoring run', alerts, top_n=5))

# Temperature monitoring run

## Alerts

- Total: 2
- Critical: 2

### Top 2 alerts

- **CRITICAL** `temperature` @ 2024-01-03T10:00:00 — CRITICAL: robust_zscore flagged sensor temperature (score 29.66)
    - modified z-score +29.66 exceeds 5
- **CRITICAL** `temperature` @ 2024-01-05T02:30:00 — CRITICAL: robust_zscore flagged sensor temperature (score 16.44)
    - modified z-score -16.44 exceeds 5



## Takeaways

- A rolling point detector is **blinded by seasonality**; deseasonalising first lets the **robust z-score** recover the injected spikes with perfect precision and recall here.
- **CUSUM** catches sustained regime shifts that point detectors miss, reporting one event per change.
- **PSI** rises monotonically with distributional shift — a practical, scipy-free drift alarm.
- **AlertPolicy** converts raw detections into a small set of ranked, explained alerts ready for an operator or an on-call queue.

This completes the path from raw signal to actionable, explained intelligence; notebooks 5–7 take it to streaming, the full fleet, and tracked experiments.